# Bi-directional Recurrent Neural Network Example

Xây dựng mạng RNN 2 chiều với PyTorch

## Tổng quan về BiRNN

<img src="https://www.easy-tensorflow.com/images/NN/03.png" alt="nn" style="width: 600px;"/>


## Tổng quan về bộ dữ liệu MNIST

Ví dụ này sử dụng bộ dữ liệu về chữ số viết tay MNIST. Bộ dữ liệu chữa 60k mẫu cho huấn luyện và 10k mẫu cho kiểm thử.

![MNIST Dataset](https://i1.wp.com/varianceexplained.org/images/mnist.png?w=456)

Để phân loại hình ảnh sử dụng RNN, chúng ta sẽ coi mỗi hàng là 1 chuỗi pixels. Bởi vì kích thước ảnh là 28*28px, ta sẽ sử lý 28 chuỗi của 28 timesteps cho tất cả các sample.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.autograd import Variable
import numpy as np

In [ ]:
# Chuẩn bị dữ liệu
from tensorflow.keras.datasets import mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()
# Chuyển đổi sang định dạng float32.
x_train, x_test = np.array(x_train, np.float32), np.array(x_test, np.float32)
x_train, x_test = x_train.reshape([-1, 28, 28]), x_test.reshape([-1, 28, 28])
# Chuẩn hóa ảnh từ from [0, 255] to [0, 1].
x_train, x_test = x_train / 255., x_test / 255.
x_train, x_test, y_train, y_test = torch.from_numpy(x_train), torch.from_numpy(x_test), torch.from_numpy(y_train).type(torch.LongTensor), torch.from_numpy(y_test).type(torch.LongTensor)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
train_data = []
batch_size = 64
for (i, j) in zip(x_train, y_train):
    train_data.append([i, j])
trainloader = torch.utils.data.DataLoader(train_data, shuffle=True, batch_size=batch_size)

test_data = []
for (i, j) in zip(x_test, y_test):
    test_data.append([i, j])
testloader = torch.utils.data.DataLoader(test_data, shuffle=False, batch_size=batch_size)

In [ ]:
# Khởi tạo mô hình BiRNN
class BiRNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(BiRNNModel, self).__init__()

        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim

        # RNN - Bidirectional=True doubles the output features
        self.rnn = nn.RNN(input_dim, hidden_dim, layer_dim, batch_first=True, nonlinearity='relu', bidirectional=True)

        # Readout layer - Since it is bidirectional, we have hidden_dim * 2
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        # Khởi tạo hidden state
        h0 = torch.zeros(self.layer_dim * 2, x.size(0), self.hidden_dim).to(x.device)

        # Forward pass
        out, hn = self.rnn(x, h0)

        # Get the last time step output
        out = self.fc(out[:, -1, :])
        return out

In [ ]:
input_dim = 28
hidden_dim = 100
layer_dim = 1
output_dim = 10

model = BiRNNModel(input_dim, hidden_dim, layer_dim, output_dim)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
for epoch in range(5):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if i % 100 == 99:
            print('[%d, %5d] loss: %.3f' % (epoch + 1, i + 1, running_loss / 100))
            running_loss = 0.0
print('Finished Training')

[1,   100] loss: 1.945
[1,   200] loss: 1.334
[1,   300] loss: 1.151
[1,   400] loss: 0.915
[1,   500] loss: 0.817
[1,   600] loss: 0.698
[1,   700] loss: 0.606
[1,   800] loss: 0.608
[1,   900] loss: 0.529
[2,   100] loss: 0.502
[2,   200] loss: 0.469
[2,   300] loss: 0.457
[2,   400] loss: 0.416
[2,   500] loss: 0.376
[2,   600] loss: 0.387
[2,   700] loss: 0.351
[2,   800] loss: 0.336
[2,   900] loss: 0.318
[3,   100] loss: 0.300
[3,   200] loss: 0.313
[3,   300] loss: 0.298
[3,   400] loss: 0.276
[3,   500] loss: 0.272
[3,   600] loss: 0.253
[3,   700] loss: 0.255
[3,   800] loss: 0.239
[3,   900] loss: 0.263
[4,   100] loss: 0.228
[4,   200] loss: 0.239
[4,   300] loss: 0.221
[4,   400] loss: 0.220
[4,   500] loss: 0.199
[4,   600] loss: 0.195
[4,   700] loss: 0.201
[4,   800] loss: 0.201
[4,   900] loss: 0.204
[5,   100] loss: 0.203
[5,   200] loss: 0.189
[5,   300] loss: 0.189
[5,   400] loss: 0.176
[5,   500] loss: 0.165
[5,   600] loss: 0.193
[5,   700] loss: 0.170
[5,   800] 

In [ ]:
correct = 0
total = 0
# Test the model
model.eval()
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Accuracy of the network on the 10000 test images: %d %%' % (100 * correct / total))

Accuracy of the network on the 10000 test images: 94 %


## Detailed Insights and Analysis

### 1. Bi-directional Recurrent Neural Networks (BiRNN)
In a standard RNN, the model only processes information from the past (previous time steps) to predict the current state. In contrast, a **Bi-directional RNN** processes the input sequence in two directions:
- **Forward Pass:** Processes the sequence from $t=1$ to $t=T$.
- **Backward Pass:** Processes the sequence from $t=T$ to $t=1$.

By combining these two hidden states, the model gains access to both past and future context at any given point in the sequence. For image data like MNIST, this means the network can understand pixel dependencies from both the top-down and bottom-up perspectives simultaneously.

### 2. MNIST as a Sequence
In this exercise, we treat the $28 \times 28$ pixel image as a sequence of 28 rows, where each row contains 28 pixels (the features).
- **Input Dimension:** 28 (pixels per row)
- **Time Steps (Sequence Length):** 28 (number of rows)

### 3. Model Architecture Details
- **Hidden Dimension:** We used 100 units. Since it is bi-directional, the concatenated output for each time step has a dimension of 200 ($100 \times 2$).
- **Linear Layer:** The final fully connected layer takes the output of the very last time step from both directions to classify the image into one of 10 categories (digits 0-9).
- **Optimizer:** We used the **Adam optimizer**, which generally converges faster than standard SGD for recurrent architectures due to its adaptive learning rate.

### 4. Performance Expectations
Using a BiRNN typically results in higher accuracy compared to a uni-directional RNN because the model captures more spatial context. However, it requires twice the computational resources for the recurrent layers due to the dual-pass nature of the architecture.